In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

In [4]:
df = pd.read_csv('data/kmeansdata2025.csv')
df['Apprehension Date'] = pd.to_datetime(df['Apprehension Date'])
df = df[df['Apprehension Date'] >= '2025-01-20']
df['Age'] = df['Apprehension Date'].dt.year - df['Birth Year'].dropna().astype(int)
df['Age Group'] = pd.cut(df['Age'], 
                                     bins=[0, 18, 25, 35, 45, 55, 65, 100], 
                                     labels=['0-17', '18-24', '25-34', '35-44', '45-54', '55-64', '65+'])
del df['Unique Identifier']



In [6]:
def create_aor_share_matrix(df):
    """Create a DataFrame with AORs as rows and share percentages as columns"""
    
    aor_profiles = []
    
    for aor in df['aor_nam'].unique():
        if pd.notna(aor):
            # Filter data for this AOR
            aor_data = df[df['aor_nam'] == aor].copy()
            total = len(aor_data)
            
            profile = {'AOR': aor, 'Total_Arrests': total}
            
            # Calculate shares for Apprehension Criminality
            criminality_counts = aor_data['Apprehension Criminality'].value_counts()
            for category in df['Apprehension Criminality'].unique():
                if pd.notna(category):
                    count = criminality_counts.get(category, 0)
                    profile[f'Share_Criminality_{category}'] = round(count / total * 100, 2)
            
            # Calculate shares for Age Group
            age_counts = aor_data['Age Group'].value_counts()
            for category in df['Age Group'].unique():
                if pd.notna(category):
                    count = age_counts.get(category, 0)
                    profile[f'Share_AgeGroup_{category}'] = round(count / total * 100, 2)
            
            # Calculate shares for Gender
            gender_counts = aor_data['Gender'].value_counts()
            for category in df['Gender'].unique():
                if pd.notna(category):
                    count = gender_counts.get(category, 0)
                    profile[f'Share_Gender_{category}'] = round(count / total * 100, 2)
            
            # Calculate shares for top Citizenship Countries
            citizenship_counts = aor_data['Citizenship Country'].value_counts()
            # Get top 10 countries overall
            top_countries = df['Citizenship Country'].value_counts().head(10).index
            for country in top_countries:
                count = citizenship_counts.get(country, 0)
                profile[f'Share_Citizenship_{country}'] = round(count / total * 100, 2)
            
            aor_profiles.append(profile)
    
    # Create DataFrame
    aor_share_df = pd.DataFrame(aor_profiles)
    
    # Sort by AOR name
    aor_share_df = aor_share_df.sort_values('AOR').reset_index(drop=True)
    
    # Display
    print("\n" + "="*80)
    print("AOR ARREST SHARE MATRIX")
    print("="*80)
    print(f"\nDimensions: {len(aor_share_df)} AORs x {len(aor_share_df.columns)-1} features")
    print("\nFirst few rows:")
    print(aor_share_df.head().to_string(index=False))
    
    # Save
    aor_share_df.to_csv('aor_arrest_shares_matrix.csv', index=False)
    print("\n✅ Saved to: aor_arrest_shares_matrix.csv")
    
    return aor_share_df

# Create the matrix
aor_shares = create_aor_share_matrix(df)

# Display summary statistics
print("\n" + "="*80)
print("COLUMN NAMES IN THE MATRIX")
print("="*80)
print("\nCriminality columns:")
criminality_cols = [col for col in aor_shares.columns if 'Criminality' in col]
for col in criminality_cols:
    print(f"  • {col}")

print("\nAge Group columns:")
age_cols = [col for col in aor_shares.columns if 'AgeGroup' in col]
for col in age_cols:
    print(f"  • {col}")

print("\nGender columns:")
gender_cols = [col for col in aor_shares.columns if 'Gender' in col]
for col in gender_cols:
    print(f"  • {col}")

print("\nCitizenship columns:")
citizenship_cols = [col for col in aor_shares.columns if 'Citizenship' in col]
for col in citizenship_cols[:5]:  # Show first 5
    print(f"  • {col}")
print(f"  ... and {len(citizenship_cols)-5} more")

# Optional: Create a more readable version with shorter column names
print("\n" + "="*80)
print("CREATING SIMPLIFIED VERSION WITH SHORT COLUMN NAMES")
print("="*80)

aor_shares_simple = aor_shares.copy()

# Rename columns to be shorter
rename_dict = {}
for col in aor_shares_simple.columns:
    if col.startswith('Share_'):
        # Remove 'Share_' prefix and shorten
        new_name = col.replace('Share_Criminality_', 'Crim_')
        new_name = new_name.replace('Share_AgeGroup_', 'Age_')
        new_name = new_name.replace('Share_Gender_', 'Gender_')
        new_name = new_name.replace('Share_Citizenship_', 'Country_')
        rename_dict[col] = new_name

aor_shares_simple.rename(columns=rename_dict, inplace=True)
aor_shares_simple.to_csv('aor_arrest_shares_matrix_simple.csv', index=False)

print("✅ Saved simplified version to: aor_arrest_shares_matrix_simple.csv")

print("\nExample of simplified column names:")
print(list(aor_shares_simple.columns)[:10])



AOR ARREST SHARE MATRIX

Dimensions: 26 AORs x 24 features

First few rows:
      AOR  Total_Arrests  Share_Criminality_No Criminal Charges  Share_Criminality_Convicted  Share_Criminality_Pending Charges  Share_AgeGroup_18-24  Share_AgeGroup_25-34  Share_AgeGroup_35-44  Share_AgeGroup_55-64  Share_AgeGroup_45-54  Share_AgeGroup_0-17  Share_AgeGroup_65+  Share_Gender_Male  Share_Gender_Female  Share_Gender_Unknown  Share_Citizenship_MEXICO  Share_Citizenship_GUATEMALA  Share_Citizenship_HONDURAS  Share_Citizenship_VENEZUELA  Share_Citizenship_EL SALVADOR  Share_Citizenship_NICARAGUA  Share_Citizenship_COLOMBIA  Share_Citizenship_ECUADOR  Share_Citizenship_CUBA  Share_Citizenship_DOMINICAN REPUBLIC
  Atlanta          15185                                  22.67                        36.77                              40.56                 18.79                 36.91                 27.63                  2.61                 12.04                 1.73                0.28              8

In [ ]:


# ============================================================================
# STEP 1: CREATE THE SHARE MATRIX (if not already done)
# ============================================================================

def create_aor_share_matrix(df):
    """Create a DataFrame with AORs as rows and share percentages as columns"""
    
    aor_profiles = []
    
    for aor in df['aor_nam'].unique():
        if pd.notna(aor):
            aor_data = df[df['aor_nam'] == aor].copy()
            total = len(aor_data)
            
            profile = {'AOR': aor, 'Total_Arrests': total}
            
            # Criminality shares
            criminality_counts = aor_data['Apprehension Criminality'].value_counts()
            for category in df['Apprehension Criminality'].unique():
                if pd.notna(category):
                    count = criminality_counts.get(category, 0)
                    profile[f'Share_Criminality_{category}'] = round(count / total * 100, 2)
            
            # Age Group shares
            age_counts = aor_data['Age Group'].value_counts()
            for category in df['Age Group'].unique():
                if pd.notna(category):
                    count = age_counts.get(category, 0)
                    profile[f'Share_AgeGroup_{category}'] = round(count / total * 100, 2)
            
            # Gender shares
            gender_counts = aor_data['Gender'].value_counts()
            for category in df['Gender'].unique():
                if pd.notna(category):
                    count = gender_counts.get(category, 0)
                    profile[f'Share_Gender_{category}'] = round(count / total * 100, 2)
            
            # Top Citizenship Countries
            citizenship_counts = aor_data['Citizenship Country'].value_counts()
            top_countries = df['Citizenship Country'].value_counts().head(10).index
            for country in top_countries:
                count = citizenship_counts.get(country, 0)
                profile[f'Share_Citizenship_{country}'] = round(count / total * 100, 2)
            
            aor_profiles.append(profile)
    
    aor_share_df = pd.DataFrame(aor_profiles)
    aor_share_df = aor_share_df.sort_values('AOR').reset_index(drop=True)
    
    return aor_share_df

print("Creating AOR share matrix...")
aor_shares = create_aor_share_matrix(df)
print(f"✅ Created matrix: {len(aor_shares)} AORs x {len(aor_shares.columns)} columns")

# Save the matrix
aor_shares.to_csv('data/aor_share_matrix_for_clustering.csv', index=False)

# ============================================================================
# STEP 2: PREPARE DATA FOR K-MEANS
# ============================================================================

print("\n" + "="*80)
print("PREPARING DATA FOR K-MEANS CLUSTERING")
print("="*80)

# Separate AOR names from features
aor_names = aor_shares['AOR'].values
total_arrests = aor_shares['Total_Arrests'].values

# Get only the share columns (exclude AOR name and Total_Arrests)
feature_cols = [col for col in aor_shares.columns if col.startswith('Share_')]
X = aor_shares[feature_cols].values

print(f"\nNumber of AORs to cluster: {len(aor_names)}")
print(f"Number of features: {len(feature_cols)}")
print(f"\nFeatures being used:")
for i, col in enumerate(feature_cols[:10], 1):
    print(f"  {i}. {col}")
if len(feature_cols) > 10:
    print(f"  ... and {len(feature_cols)-10} more")

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\n✅ Data standardized and ready for clustering")

# ============================================================================
# STEP 3: FIND OPTIMAL NUMBER OF CLUSTERS (ELBOW METHOD)
# ============================================================================

print("\n" + "="*80)
print("FINDING OPTIMAL NUMBER OF CLUSTERS")
print("="*80)

max_k = min(10, len(aor_names) - 1)  # Can't have more clusters than AORs
inertias = []
silhouette_scores = []
K_range = range(2, max_k + 1)

print(f"\nTesting k from 2 to {max_k}...")

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    
    sil_score = silhouette_score(X_scaled, kmeans.labels_)
    silhouette_scores.append(sil_score)
    print(f"  k={k}: Inertia={kmeans.inertia_:.2f}, Silhouette={sil_score:.3f}")

# Plot elbow curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Elbow plot
ax1.plot(list(K_range), inertias, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Number of Clusters (k)', fontsize=12)
ax1.set_ylabel('Inertia', fontsize=12)
ax1.set_title('Elbow Method - AOR Clustering', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Silhouette plot
ax2.plot(list(K_range), silhouette_scores, 'ro-', linewidth=2, markersize=8)
ax2.set_xlabel('Number of Clusters (k)', fontsize=12)
ax2.set_ylabel('Silhouette Score', fontsize=12)
ax2.set_title('Silhouette Score - AOR Clustering', fontsize=14, fontweight='bold')
ax2.axhline(y=0.5, color='g', linestyle='--', label='Good threshold (0.5)')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.savefig('data/elbow_analysis_aor_share_clustering.png', dpi=300, bbox_inches='tight')
plt.close()
print("\n✅ Saved: elbow_analysis_aor_share_clustering.png")

# Choose optimal k
optimal_k = list(K_range)[np.argmax(silhouette_scores)]
print(f"\n🎯 Optimal k based on silhouette score: {optimal_k}")

# ============================================================================
# STEP 4: PERFORM K-MEANS WITH OPTIMAL K
# ============================================================================

print("\n" + "="*80)
print(f"PERFORMING K-MEANS CLUSTERING WITH K={optimal_k}")
print("="*80)

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

# Add cluster labels to the dataframe
aor_shares['Cluster'] = cluster_labels

# Calculate silhouette score
final_sil_score = silhouette_score(X_scaled, cluster_labels)
print(f"\nFinal Silhouette Score: {final_sil_score:.3f}")

# ============================================================================
# STEP 5: ANALYZE EACH CLUSTER
# ============================================================================

print("\n" + "="*80)
print("CLUSTER ANALYSIS")
print("="*80)

cluster_analysis = []

for cluster_id in range(optimal_k):
    cluster_aors = aor_shares[aor_shares['Cluster'] == cluster_id]
    cluster_size = len(cluster_aors)
    
    print(f"\n🏷️  Cluster {cluster_id} ({cluster_size} AORs)")
    print("-" * 60)
    
    # List AORs in this cluster
    aor_list = cluster_aors['AOR'].tolist()
    print(f"AORs: {', '.join(aor_list)}")
    
    # Calculate average profile for this cluster
    cluster_means = cluster_aors[feature_cols].mean()
    
    # Find top 5 distinguishing features
    print("\n📊 Top distinguishing characteristics:")
    
    # Compare to overall average
    overall_means = aor_shares[feature_cols].mean()
    differences = cluster_means - overall_means
    top_differences = differences.abs().nlargest(5)
    
    for feature in top_differences.index:
        cluster_val = cluster_means[feature]
        overall_val = overall_means[feature]
        diff = differences[feature]
        
        # Clean up feature name for display
        feature_display = feature.replace('Share_Criminality_', 'Criminality: ')
        feature_display = feature_display.replace('Share_AgeGroup_', 'Age: ')
        feature_display = feature_display.replace('Share_Gender_', 'Gender: ')
        feature_display = feature_display.replace('Share_Citizenship_', 'Country: ')
        
        if diff > 0:
            print(f"  • HIGH {feature_display}: {cluster_val:.1f}% (avg: {overall_val:.1f}%, +{diff:.1f}%)")
        else:
            print(f"  • LOW {feature_display}: {cluster_val:.1f}% (avg: {overall_val:.1f}%, {diff:.1f}%)")
    
    # Store for export
    cluster_analysis.append({
        'Cluster': cluster_id,
        'Num_AORs': cluster_size,
        'AORs': ', '.join(aor_list),
        'Avg_Total_Arrests': cluster_aors['Total_Arrests'].mean()
    })

# Save cluster assignments
aor_shares.to_csv('data/aor_clusters_from_shares.csv', index=False)
print("\n✅ Saved cluster assignments to: data/aor_clusters_from_shares.csv")

# ============================================================================
# STEP 6: CREATE SUMMARY FOR MAPPING
# ============================================================================

print("\n" + "="*80)
print("SUMMARY FOR MAP COLOR-CODING")
print("="*80)

map_summary = aor_shares[['AOR', 'Cluster', 'Total_Arrests']].copy()
map_summary = map_summary.sort_values('Cluster').reset_index(drop=True)

print("\nAORs grouped by cluster (use for map colors):")
print(map_summary.to_string(index=False))

map_summary.to_csv('data/aor_map_color_groups.csv', index=False)
print("\n✅ Saved: aor_map_color_groups.csv")

# Create a color assignment suggestion
print("\n🎨 SUGGESTED COLOR SCHEME FOR MAP:")
colors = ['Red', 'Blue', 'Green', 'Orange', 'Purple', 'Yellow', 'Pink', 'Brown', 'Gray', 'Cyan']
for cluster_id in range(optimal_k):
    cluster_aors = map_summary[map_summary['Cluster'] == cluster_id]['AOR'].tolist()
    color = colors[cluster_id % len(colors)]
    print(f"  Cluster {cluster_id} ({color}): {', '.join(cluster_aors)}")

# ============================================================================
# STEP 7: FINAL SUMMARY
# ============================================================================

print("\n" + "="*80)
print("CLUSTERING COMPLETE!")
print("="*80)

print(f"\n📊 Summary Statistics:")
print(f"  • Total AORs clustered: {len(aor_names)}")
print(f"  • Number of clusters: {optimal_k}")
print(f"  • Clustering quality (silhouette): {final_sil_score:.3f}")
print(f"  • Features used: {len(feature_cols)}")

print("\n📁 Files Generated:")
print("  • aor_share_matrix_for_clustering.csv - Original share data")
print("  • aor_clusters_from_shares.csv - AORs with cluster assignments")
print("  • aor_map_color_groups.csv - Simple mapping file")
print("  • elbow_analysis_aor_share_clustering.png - Elbow curves")

print("\n🗺️  For Your Map:")
print("  Use 'aor_map_color_groups.csv' to assign colors")
print("  All AORs with the same Cluster number get the same color")

Creating AOR share matrix...
✅ Created matrix: 26 AORs x 25 columns

PREPARING DATA FOR K-MEANS CLUSTERING

Number of AORs to cluster: 26
Number of features: 23

Features being used:
  1. Share_Criminality_No Criminal Charges
  2. Share_Criminality_Convicted
  3. Share_Criminality_Pending Charges
  4. Share_AgeGroup_18-24
  5. Share_AgeGroup_25-34
  6. Share_AgeGroup_35-44
  7. Share_AgeGroup_55-64
  8. Share_AgeGroup_45-54
  9. Share_AgeGroup_0-17
  10. Share_AgeGroup_65+
  ... and 13 more

✅ Data standardized and ready for clustering

FINDING OPTIMAL NUMBER OF CLUSTERS

Testing k from 2 to 10...
  k=2: Inertia=476.78, Silhouette=0.179
  k=3: Inertia=379.36, Silhouette=0.205
  k=4: Inertia=321.45, Silhouette=0.235
  k=5: Inertia=285.88, Silhouette=0.230
  k=6: Inertia=254.67, Silhouette=0.213
  k=7: Inertia=230.57, Silhouette=0.161
  k=8: Inertia=201.81, Silhouette=0.182
  k=9: Inertia=173.48, Silhouette=0.192
  k=10: Inertia=154.68, Silhouette=0.184

✅ Saved: elbow_analysis_aor_share

In [20]:
aorclusters = pd.read_csv('data/aor_map_color_groups.csv')

In [24]:
aorclusters['AOR'] = aorclusters['AOR'].replace('St. Paul', 'St Paul')

In [25]:
aorclusters

,AOR,Cluster,Total_Arrests
0,Boston,0,6843
1,Buffalo,0,2221
2,Philadelphia,0,6183
3,Newark,0,5976
4,New York City,0,5112
5,Atlanta,1,15185
6,Phoenix,1,6890
7,New Orleans,1,17243
8,Miami,1,23197
9,St Paul,1,4915


In [26]:
aorclusters.to_csv('data/aor_map_color_groups.csv', index=False)